# 2. Europe vs the Middle East

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hpa-code/simcitynet/blob/main/docs/tutorials/02_europe_vs_middle_east.ipynb)

*Run it in Colab (free GPU) to see every figure; the website shows the saved outputs.*

The SimCityNet paper compares street scenes in seven European and fourteen Middle Eastern
cities. This notebook runs a miniature version of that comparison on six openly licensed
photos — three per region — to show the workflow. Six photos say nothing about the regions
themselves; the paper used more than 600,000 images.

In [ ]:
# Installs simcitynet with all models (PyTorch, Transformers, Ultralytics YOLO).
# In Colab, choose Runtime > Change runtime type > GPU for speed.
%pip install -q "simcitynet[all]"
# Before the first PyPI release, install from GitHub instead:
# %pip install -q "simcitynet[all] @ git+https://github.com/hpa-code/simcitynet.git"

In [ ]:
import pandas as pd
from simcitynet import SimCityNet
from simcitynet.datasets import EXAMPLES, fetch_examples, credits_table

paths = fetch_examples("examples")             # amsterdam, dublin, zurich, doha, dubai, tel_aviv
region = pd.Series({e.name: e.region for e in EXAMPLES}, name="region")
pd.DataFrame(credits_table("examples"))[["name", "city", "region", "author", "license"]]

In [ ]:
scn = SimCityNet()
emb = scn.embed(paths)
summary = emb.summary().join(region)
summary.round(2)

## The six scenes at a glance

In [ ]:
for name in emb.names:
    display(emb.plot.scene_card(name))

## Region averages

Average of the eight summary quantities per region:

In [ ]:
summary.groupby("region").mean().round(2).T

## Which scenes look alike?

The heatmap uses all 33 dimensions. The map places images so that similar scenes sit close
together (principal components of the normalised vectors).

In [ ]:
emb.plot.heatmap()

In [ ]:
emb.plot.map()

## Compare one block at a time

Because every dimension is interpretable, you can ask *in what way* scenes are similar.
Structure only (vegetation and surfaces):

In [ ]:
emb.plot.heatmap(blocks=["vegetation", "surfaces"])

Activity only (pedestrians and vehicles):

In [ ]:
emb.plot.heatmap(blocks=["pedestrians", "vehicles"])

## Colour palettes within and between regions

The paper found that AI-generated scenes compress colour variety. Here we measure the average
Earth Mover's distance between palettes within each region and across regions.

In [ ]:
import numpy as np

emd = emb.distance("color_emd")
groups = region.reindex(emd.index)

def mean_between(a, b):
    block = emd.loc[groups == a, groups == b].to_numpy()
    if a == b:
        block = block[~np.eye(len(block), dtype=bool)]
    return float(block.mean())

pd.DataFrame(
    {g: {h: mean_between(g, h) for h in ["Europe", "Middle East"]} for g in ["Europe", "Middle East"]}
).round(1)

In [ ]:
for name in emb.names:
    display(emb.plot.palette(name))

## Radar profiles by region

In [ ]:
europe = [n for n in emb.names if region[n] == "Europe"]
middle_east = [n for n in emb.names if region[n] == "Middle East"]
display(emb.plot.radar(europe))
display(emb.plot.radar(middle_east))

## Nearest neighbours

In [ ]:
emb.plot.neighbors("amsterdam", k=3)

## Going further

- Embed AI-generated frames for the same places (see **[3. Video frames](03_video_frames.ipynb)**)
  and compare each frame with the real photos.
- For statistical comparisons like the paper's, export `emb.raw` and use SciPy or statsmodels
  (e.g. per-block Mahalanobis distances and Mann–Whitney tests).
- Figures made from CC BY-SA photos must credit the authors (`CREDITS.csv`) and be shared
  under the same licence.